Vorige week heb je met behulp van Keras een neuraal netwerk gebouwd voor de MNIST‑dataset.


Dit netwerk gebruikte ingebouwde activatiefuncties zoals relu en softmax.



Deze week ga je:

- Activatiefuncties zelf implementeren
- Begrijpen wat pre‑activatie, logits en outputinterpretatie zijn
- Onderzoeken wat het effect van verschillende activatiefuncties is op het gedrag van je netwerk

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import OneHotEncoder

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
X_train_flat = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test_flat  = X_test.reshape(-1, 784).astype('float32') / 255.0

encoder = OneHotEncoder(sparse_output=False)
y_train_ohe = encoder.fit_transform(y_train.reshape(-1, 1))
y_test_ohe  = encoder.transform(y_test.reshape(-1, 1))

print(f"Trainset: {X_train_flat.shape}  |  Testset: {X_test_flat.shape}")

### Pre‑activatie expliciet maken

Normaal voert Keras deze stap automatisch uit. In deze opdracht ga je dit zichtbaar maken.

#### Pre-Activation
Gegeven één neuron met:

inputs: x
gewichten: w
bias: b


Schrijf in Python een functie:

def ***pre_activation(x, w, b)***. Deze functie moet de pre‑activatie berekenen (dus nog zonder activatiefunctie).

Test je functie met zelfgekozen waarden.
Print de uitkomst en leg in commentaar uit:

wat deze waarde betekent
waarom dit nog geen output van het neuron is

In [ ]:
def pre_activation(x, w, b):
    return np.dot(w, x) + b

x = np.array([0.5, 0.3, 0.2])
w = np.array([0.4, 0.6, 0.8])
b = 0.1
z = pre_activation(x, w, b)
print(f"Pre-activatie z = {z:.4f}")
# z is de gewogen som van de inputs plus de bias.
# Dit is nog geen output van het neuron: de activatiefunctie ontbreekt nog.
# De activatiefunctie bepaalt pas of en hoe sterk het neuron "vuurt".

#### Activatiefuncties zelf maken

Tot nu toe heb je default activatiefunctie implementaties gebruikt, maat laten we nu kijken naar eigen implementatie om te begrijpen hoe het nou werkt. (Mogelijk heb je enkele van de functies al gemaakt bij de CodeGrade opdrachten van deze week.)

#### ReLU zelf implementeren

Schrijf een functie:

***def relu(z):***

De functie moet werken voor:

- 1 getal
- 1 numpy array


Test je functie met:

- negatieve waarden
- nul
- positieve waarden



#### Sigmoid zelf implementeren

Zoek de formule op. en implementeer de functie:

***def sigmoid(z):***

Test de functie met:

- grote negatieve waarden
- 0
- grote positieve waarden


- Wanneer gaat de output richting 0?

  Bij **grote negatieve** waarden van z (z << 0) nadert de output naar 0.

- Wanneer richting 1?

  Bij **grote positieve** waarden van z (z >> 0) nadert de output naar 1.

- Wat betekent een output van 0.5?

  Dit treedt op wanneer z = 0. Het netwerk is volledig **onzeker**: gelijke kans voor beide klassen.


#### Softmax zelf implementeren

Zoek de formule op. en implementeer de functie:

Je krijgt de formule:

***def softmax(z):*** 

Test met verschillende lijstjes.

Let er op dat de outputs optellen tot 1.

waarom softmax ook al weer nuttig is bij multi‑class classificatie?

  Softmax zet ruwe scores (logits) om naar **kansen die optellen tot 1**. Zo geeft het netwerk per klasse een interpreteerbare waarschijnlijkheid, wat nodig is bij 10 klassen zoals in MNIST.

In [ ]:
def relu(z):
    return np.maximum(0, z)

print("--- ReLU ---")
print(f"Negatief (-3) : {relu(-3)}")
print(f"Nul (0)       : {relu(0)}")
print(f"Positief (5)  : {relu(5)}")
print(f"Array         : {relu(np.array([-2, -1, 0, 3, 5]))}")


def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print("\n--- Sigmoid ---")
print(f"Grote negatieve waarde (-100): {sigmoid(-100):.6f}")
print(f"Nul (0)                      : {sigmoid(0):.6f}")
print(f"Grote positieve waarde (100) : {sigmoid(100):.6f}")


def softmax(z):
    e = np.exp(z - np.max(z))  # aftrekken van max voor numerieke stabiliteit
    return e / e.sum()

print("\n--- Softmax ---")
z_test = np.array([1.0, 2.0, 3.0])
probs = softmax(z_test)
print(f"Input : {z_test}")
print(f"Output: {probs.round(4)}")
print(f"Som   : {probs.sum():.4f}")

#### Je eigen Activatiefuncties gebruiken in het netwerk

Maak een Keras‑model (of gebruik die uit de vorige practicum opdracht) waarin je geen activatiefunctie opgeeft in de layers
alleen Dense(units) gebruikt


- Wat zijn de outputs van de hidden layer nu?

  **Ruwe logits**: gewogen sommen van de inputs plus bias, niet begrensd in bereik (kunnen negatief of positief zijn).

- Wat stellen deze waarden conceptueel voor?

  **Pre-activatiewaarden**: de "sterkte" van elk neuron vóór enige filtering. Zonder activatiefunctie kan het netwerk alleen lineaire relaties leren.

Gebruik nu dan je zelfgeschreven activatiefuncties om:

De output van de hidden layer handmatig door ReLU te halen
De output van de laatste layer door:

- sigmoid (single‑class experiment)
- softmax (multi‑class experiment)

Hoe kun je dit in je mnist db gebruiken(single‑class en multi-class)?

  **Single-class**: stel de binaire vraag "is dit beeld een 7?" — één output node met sigmoid geeft een kans tussen 0 en 1.
  **Multi-class**: stel de vraag "welk cijfer (0–9) is dit?" — 10 output nodes met softmax geven kansen per klasse.

Welke heeft het beste gewerkt? Waarom?

  **Softmax (multi-class)** werkt beter voor MNIST. MNIST heeft 10 klassen en softmax leert alle klassen tegelijk onderscheiden. Sigmoid is geschikt voor binaire classificatie maar mist het onderscheid tussen de overige 9 cijfers.

Probeer je netwerk zo te maken dat het nog steeds op de Mystery device past.

In [ ]:
# Model zonder activatiefuncties
model_raw = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(64),
    tf.keras.layers.Dense(10),
])

raw_out = model_raw(X_test_flat[:3]).numpy()
print("Output zonder activatiefunctie:")
print(raw_out[0].round(4))
# Ruwe logits: gewogen sommen + bias, niet begrensd in bereik.
# Zonder activatiefunctie kan het netwerk alleen lineaire relaties leren.

# Eigen relu toepassen op de hidden layer output
print("\nNa eigen relu:")
print(relu(raw_out[0]).round(4))

# Eigen softmax toepassen (multi-class output)
probs = softmax(raw_out[0])
print("\nNa eigen softmax (multi-class):")
print(probs.round(4))
print(f"Som: {probs.sum():.4f}")

# Eigen sigmoid toepassen (single-class output, eerste node)
print(f"\nNa eigen sigmoid (single-class): {sigmoid(raw_out[0][0]):.4f}")

In [ ]:
# Single-class: is het een 7? (sigmoid)
y_train_7 = (y_train == 7).astype('float32')
y_test_7  = (y_test  == 7).astype('float32')

model_single = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(64),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
model_single.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_single.fit(X_train_flat, y_train_7, epochs=5, batch_size=64, verbose=1)

# Multi-class: welk cijfer? (softmax)
model_multi = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(64),
    tf.keras.layers.Dense(10, activation='softmax'),
])
model_multi.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_multi.fit(X_train_flat, y_train_ohe, epochs=5, batch_size=64, verbose=1)

_, acc_single = model_single.evaluate(X_test_flat, y_test_7, verbose=0)
_, acc_multi  = model_multi.evaluate(X_test_flat, y_test_ohe, verbose=0)
print(f"\nSingle-class met sigmoid: {acc_single:.4f}")
print(f"Multi-class  met softmax: {acc_multi:.4f}")